# Evaluate fine-tuned conditions ? Colab T4

Load the completed adapter from Drive and run the fine-tuned and fine-tuned plus RAG conditions.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Select a GPU runtime in Runtime > Change runtime type"

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, json, shutil, time

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/farmer-llm')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
REPO_URL = 'https://github.com/sarthakmehra02/stuff.git'
REPO = Path('/content/farmer-llm-project')
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
os.chdir(REPO)


In [ ]:
%pip install -r requirements-colab.txt
from pathlib import Path
ADAPTER = '/content/drive/MyDrive/farmer-llm/run1/adapter'
assert Path(ADAPTER).exists(), ADAPTER

In [ ]:
%%time
!python src/run_conditions.py --conditions finetuned,finetuned_rag --limit 5 --adapter /content/drive/MyDrive/farmer-llm/run1/adapter --out results --resume
for condition in ['finetuned','finetuned_rag']:
    print('\n###', condition)
    with open(f'results/raw_{condition}.jsonl', encoding='utf-8') as f:
        for line in f:
            row=json.loads(line); print(row['id'], row['answer'])


## Check five answers

After reviewing these examples, run the full cell.

In [ ]:
%%time
!python src/run_conditions.py --conditions finetuned,finetuned_rag --adapter /content/drive/MyDrive/farmer-llm/run1/adapter --out results --resume
for path in Path('results').glob('raw_finetuned*.jsonl'):
    shutil.copy2(path, DRIVE_ROOT / path.name)
shutil.copy2('results/run_metadata.json', DRIVE_ROOT / 'run_metadata.json')
print('Copy the raw_*.jsonl files from Drive to the local repository results/ folder for judging.')
